# 04 — Identificar afiliaciones UNAM

Este notebook identifica qué relaciones autor–afiliación corresponden a la UNAM.

Reglas principales:

- `UNAM`: se conserva.
- `EXTERNO`: se elimina de la salida.
- `REVISAR`: se guarda aparte.
- UNAM + externa: se conserva únicamente la afiliación UNAM.
- Dos afiliaciones UNAM: una se guarda en `Afiliacion1` y otra en `Afiliacion2`.
- No se modifica `Autor_norm`.
- No se modifica ningún otro campo bibliográfico.
- No se deduplican publicaciones.
- Los archivos originales nunca se sobrescriben.

El archivo `clasificacion_afiliaciones_unam.csv` contiene las decisiones institucionales ya revisadas.

In [ ]:
import pandas as pd
import os
import hashlib
from collections import Counter

# --------------------------------------------------
# Rutas del repositorio
# --------------------------------------------------

archivo = "../04_Limpieza/00_Separacion_Autor_Afiliacion/autor_afiliacion_separado.csv"

archivo_maestro = "../00_control/UNAM_Completo_Corregido.csv"

carpeta_salida = "../04_Limpieza/01_Internos_unam"

archivo_clasificacion = carpeta_salida + "/clasificacion_afiliaciones_unam.csv"

archivo_unam = carpeta_salida + "/autores_unam_identificados.csv"

archivo_revision = carpeta_salida + "/afiliaciones_revision.csv"


## 1. Cargar los archivos

Los CSV actuales contienen algunas columnas vacías adicionales al final.  
El notebook comprueba que esas columnas estén realmente vacías y después trabaja solamente con las columnas necesarias.

In [ ]:
def sha256(ruta):
    h = hashlib.sha256()

    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloque)

    return h.hexdigest()


hash_original_antes = sha256(archivo)


# Cargar archivos.
df = pd.read_csv(
    archivo,
    dtype=str,
    encoding="utf-8-sig"
).fillna("")

maestro = pd.read_csv(
    archivo_maestro,
    dtype=str,
    encoding="utf-8-sig"
).fillna("")

clasificacion = pd.read_csv(
    archivo_clasificacion,
    dtype=str,
    encoding="utf-8-sig"
).fillna("")


# Quitar espacios accidentales de los encabezados.
df.columns = df.columns.str.strip()
maestro.columns = maestro.columns.str.strip()
clasificacion.columns = clasificacion.columns.str.strip()


columnas_canonicas = [
    "Base_origen",
    "Fuente_origen",
    "indice",
    "Titulo",
    "Año",
    "Autor_norm",
    "Afiliacion1",
    "Afiliacion2",
    "ISBN",
    "ISSN",
    "Doi",
    "URL",
    "Area",
    "SubArea",
    "Keywords",
    "Abstract",
]

columnas_clasificacion = [
    "Afiliacion_original",
    "Estado_UNAM",
    "Evidencia",
]


# --------------------------------------------------
# Comprobar columnas necesarias
# --------------------------------------------------

faltantes_df = [
    columna
    for columna in columnas_canonicas
    if columna not in df.columns
]

faltantes_clasificacion = [
    columna
    for columna in columnas_clasificacion
    if columna not in clasificacion.columns
]

if faltantes_df:
    raise ValueError(
        "Faltan columnas canónicas en autor_afiliacion_separado.csv: "
        + str(faltantes_df)
    )

if faltantes_clasificacion:
    raise ValueError(
        "Faltan columnas en clasificacion_afiliaciones_unam.csv: "
        + str(faltantes_clasificacion)
    )


# --------------------------------------------------
# Comprobar columnas adicionales
# --------------------------------------------------

extras_df = [
    columna
    for columna in df.columns
    if columna not in columnas_canonicas
]

extras_clasificacion = [
    columna
    for columna in clasificacion.columns
    if columna not in columnas_clasificacion
]


# No ignorar columnas adicionales si contienen información.
if extras_df:
    if (
        df[extras_df]
        .astype(str)
        .apply(lambda columna: columna.str.strip().ne("").any())
        .any()
    ):
        raise ValueError(
            "Hay columnas adicionales con información en "
            "autor_afiliacion_separado.csv"
        )

if extras_clasificacion:
    if (
        clasificacion[extras_clasificacion]
        .astype(str)
        .apply(lambda columna: columna.str.strip().ne("").any())
        .any()
    ):
        raise ValueError(
            "Hay columnas adicionales con información en "
            "clasificacion_afiliaciones_unam.csv"
        )


# Trabajar únicamente con las columnas necesarias.
df = df[columnas_canonicas].copy()

clasificacion = clasificacion[
    columnas_clasificacion
].copy()


# El archivo de clasificación actual tiene una fila final vacía.
# Se elimina solamente de la copia en memoria.
clasificacion = clasificacion[
    clasificacion["Afiliacion_original"].str.strip() != ""
].copy()

clasificacion = clasificacion.reset_index(drop=True)


print("Relaciones cargadas:", len(df))
print("Columnas canónicas utilizadas:", len(df.columns))
print("Afiliaciones clasificadas:", len(clasificacion))
print("Filas con Afiliacion2 no vacía:", (~df["Afiliacion2"].eq("")).sum())
print("Columnas vacías adicionales ignoradas en entrada:", len(extras_df))
print(
    "Columnas vacías adicionales ignoradas en clasificación:",
    len(extras_clasificacion)
)


Relaciones cargadas: 9515
Columnas canónicas utilizadas: 16
Afiliaciones clasificadas: 1604
Filas con Afiliacion2 no vacía: 0
Columnas vacías adicionales ignoradas en entrada: 4
Columnas vacías adicionales ignoradas en clasificación: 17


## 2. Validar el diccionario de clasificación

Se comprueba que cada afiliación de `autor_afiliacion_separado.csv` tenga exactamente una clasificación.

In [ ]:
estados_validos = {
    "UNAM",
    "EXTERNO",
    "REVISAR"
}


# Cada afiliación debe aparecer una sola vez en el diccionario.
duplicadas = clasificacion[
    clasificacion["Afiliacion_original"].duplicated(keep=False)
]

if len(duplicadas) > 0:
    display(duplicadas)

    raise ValueError(
        "Existen afiliaciones duplicadas en el diccionario."
    )


# Solamente se permiten los tres estados definidos.
estados_invalidos = (
    set(clasificacion["Estado_UNAM"])
    - estados_validos
)

if estados_invalidos:
    raise ValueError(
        "Estados no válidos encontrados: "
        + str(estados_invalidos)
    )


# Comparar afiliaciones del archivo con el diccionario.
afiliaciones_entrada = set(df["Afiliacion1"])

afiliaciones_diccionario = set(
    clasificacion["Afiliacion_original"]
)

faltantes = sorted(
    afiliaciones_entrada
    - afiliaciones_diccionario
)

sobrantes = sorted(
    afiliaciones_diccionario
    - afiliaciones_entrada
)


print("Afiliaciones únicas en entrada:", len(afiliaciones_entrada))
print("Afiliaciones únicas en diccionario:", len(afiliaciones_diccionario))
print("Afiliaciones sin clasificación:", len(faltantes))
print("Afiliaciones del diccionario no presentes en entrada:", len(sobrantes))


if faltantes:
    print("\nPrimeras afiliaciones sin clasificación:")

    display(
        pd.DataFrame(
            faltantes[:20],
            columns=["Afiliacion_original"]
        )
    )

    raise ValueError(
        "Hay afiliaciones que todavía no están clasificadas."
    )


Afiliaciones únicas en entrada: 1604
Afiliaciones únicas en diccionario: 1604
Afiliaciones sin clasificación: 0
Afiliaciones del diccionario no presentes en entrada: 0


## 3. Ajustes exactos para afiliaciones múltiples

La mayoría de las afiliaciones UNAM no requiere ningún cambio.

Este diccionario contiene únicamente los casos que fueron revisados manualmente porque:

- tenían UNAM + una institución externa;
- contenían dos afiliaciones UNAM;
- o la fuente bibliográfica unió varias instituciones en una sola cadena.

Las comparaciones son exactas. No se usa fuzzy matching.

In [ ]:
ajustes_afiliaciones = {'BioRobotics Laboratory, School of Engineering, National Autonomous University of Mexico, Circuito Exterior S/N, Ciudad Universitaria, Coyoacán, Mexico City, 04510, Mexico, Telecommunications Department, School of Electrical Engineering, Central University of Venezuela, Ciudad Universitaria de Caracas, Los Chaguaramos, Caracas, 1051, Venezuela': ('BioRobotics '
                                                                                                                                                                                                                                                                                                                                                              'Laboratory, '
                                                                                                                                                                                                                                                                                                                                                              'School '
                                                                                                                                                                                                                                                                                                                                                              'of '
                                                                                                                                                                                                                                                                                                                                                              'Engineering, '
                                                                                                                                                                                                                                                                                                                                                              'National '
                                                                                                                                                                                                                                                                                                                                                              'Autonomous '
                                                                                                                                                                                                                                                                                                                                                              'University '
                                                                                                                                                                                                                                                                                                                                                              'of '
                                                                                                                                                                                                                                                                                                                                                              'Mexico, '
                                                                                                                                                                                                                                                                                                                                                              'Circuito '
                                                                                                                                                                                                                                                                                                                                                              'Exterior '
                                                                                                                                                                                                                                                                                                                                                              'S/N, '
                                                                                                                                                                                                                                                                                                                                                              'Ciudad '
                                                                                                                                                                                                                                                                                                                                                              'Universitaria, '
                                                                                                                                                                                                                                                                                                                                                              'Coyoacán, '
                                                                                                                                                                                                                                                                                                                                                              'Mexico '
                                                                                                                                                                                                                                                                                                                                                              'City, '
                                                                                                                                                                                                                                                                                                                                                              '04510, '
                                                                                                                                                                                                                                                                                                                                                              'Mexico',
                                                                                                                                                                                                                                                                                                                                                              ''),
 'Centro de Ciencias de la Complejidad, Universidad Nacional Autónoma de México, Mexico City, 04510, Mexico, Posgrado en Ciencias de la Computación, Universidad Nacional Autónoma de México, Mexico City, 04510, Mexico': ('Centro '
                                                                                                                                                                                                                            'de '
                                                                                                                                                                                                                            'Ciencias '
                                                                                                                                                                                                                            'de '
                                                                                                                                                                                                                            'la '
                                                                                                                                                                                                                            'Complejidad, '
                                                                                                                                                                                                                            'Universidad '
                                                                                                                                                                                                                            'Nacional '
                                                                                                                                                                                                                            'Autónoma '
                                                                                                                                                                                                                            'de '
                                                                                                                                                                                                                            'México, '
                                                                                                                                                                                                                            'Mexico '
                                                                                                                                                                                                                            'City, '
                                                                                                                                                                                                                            '04510, '
                                                                                                                                                                                                                            'Mexico',
                                                                                                                                                                                                                            'Posgrado '
                                                                                                                                                                                                                            'en '
                                                                                                                                                                                                                            'Ciencias '
                                                                                                                                                                                                                            'de '
                                                                                                                                                                                                                            'la '
                                                                                                                                                                                                                            'Computación, '
                                                                                                                                                                                                                            'Universidad '
                                                                                                                                                                                                                            'Nacional '
                                                                                                                                                                                                                            'Autónoma '
                                                                                                                                                                                                                            'de '
                                                                                                                                                                                                                            'México, '
                                                                                                                                                                                                                            'Mexico '
                                                                                                                                                                                                                            'City, '
                                                                                                                                                                                                                            '04510, '
                                                                                                                                                                                                                            'Mexico'),
 'Centro de Ciencias de la Complejidad, Universidad Nacional Autónoma de México, Mexico City, 04510, Mexico, Roslin Institute, University of Edinburgh, Midlothian, EH8 9YL, United Kingdom': ('Centro '
                                                                                                                                                                                               'de '
                                                                                                                                                                                               'Ciencias '
                                                                                                                                                                                               'de '
                                                                                                                                                                                               'la '
                                                                                                                                                                                               'Complejidad, '
                                                                                                                                                                                               'Universidad '
                                                                                                                                                                                               'Nacional '
                                                                                                                                                                                               'Autónoma '
                                                                                                                                                                                               'de '
                                                                                                                                                                                               'México, '
                                                                                                                                                                                               'Mexico '
                                                                                                                                                                                               'City, '
                                                                                                                                                                                               '04510, '
                                                                                                                                                                                               'Mexico',
                                                                                                                                                                                               ''),
 'Centro de Ciencias de la Complejidad, Universidad Nacional Autónoma de México, Mexico City, 04510, Mexico, School of Systems Science and Industrial Engineering, Binghamton University, Binghamton, 13902, NY, United States, Instituto de Investigaciones en Matemáticas Aplicadas y Sistemas, Universidad Nacional Autónoma de México, Mexico City, 04510, Mexico, Santa Fe Institute, Santa Fe, 87501, NM, United States': ('Centro '
                                                                                                                                                                                                                                                                                                                                                                                                                                 'de '
                                                                                                                                                                                                                                                                                                                                                                                                                                 'Ciencias '
                                                                                                                                                                                                                                                                                                                                                                                                                                 'de '
                                                                                                                                                                                                                                                                                                                                                                                                                                 'la '
                                                                                                                                                                                                                                                                                                                                                                                                                                 'Complejidad, '
                                                                                                                                                                                                                                                                                                                                                                                                                                 'Universidad '
                                                                                                                                                                                                                                                                                                                                                                                                                                 'Nacional '
                                                                                                                                                                                                                                                                                                                                                                                                                                 'Autónoma '
                                                                                                                                                                                                                                                                                                                                                                                                                                 'de '
                                                                                                                                                                                                                                                                                                                                                                                                                                 'México, '
                                                                                                                                                                                                                                                                                                                                                                                                                                 'Mexico '
                                                                                                                                                                                                                                                                                                                                                                                                                                 'City, '
                                                                                                                                                                                                                                                                                                                                                                                                                                 '04510, '
                                                                                                                                                                                                                                                                                                                                                                                                                                 'Mexico',
                                                                                                                                                                                                                                                                                                                                                                                                                                 'Instituto '
                                                                                                                                                                                                                                                                                                                                                                                                                                 'de '
                                                                                                                                                                                                                                                                                                                                                                                                                                 'Investigaciones '
                                                                                                                                                                                                                                                                                                                                                                                                                                 'en '
                                                                                                                                                                                                                                                                                                                                                                                                                                 'Matemáticas '
                                                                                                                                                                                                                                                                                                                                                                                                                                 'Aplicadas '
                                                                                                                                                                                                                                                                                                                                                                                                                                 'y '
                                                                                                                                                                                                                                                                                                                                                                                                                                 'Sistemas, '
                                                                                                                                                                                                                                                                                                                                                                                                                                 'Universidad '
                                                                                                                                                                                                                                                                                                                                                                                                                                 'Nacional '
                                                                                                                                                                                                                                                                                                                                                                                                                                 'Autónoma '
                                                                                                                                                                                                                                                                                                                                                                                                                                 'de '
                                                                                                                                                                                                                                                                                                                                                                                                                                 'México, '
                                                                                                                                                                                                                                                                                                                                                                                                                                 'Mexico '
                                                                                                                                                                                                                                                                                                                                                                                                                                 'City, '
                                                                                                                                                                                                                                                                                                                                                                                                                                 '04510, '
                                                                                                                                                                                                                                                                                                                                                                                                                                 'Mexico'),
 'Centro de Estudios en Computación Avanzada, Universidad Nacional Autonoma de México, Mexico City, Mexico, Departamento de Procesamiento de Señales, Facultad de Ingeniería, Universidad Nacional Autonoma de México, Mexico City, Mexico': ('Centro '
                                                                                                                                                                                                                                              'de '
                                                                                                                                                                                                                                              'Estudios '
                                                                                                                                                                                                                                              'en '
                                                                                                                                                                                                                                              'Computación '
                                                                                                                                                                                                                                              'Avanzada, '
                                                                                                                                                                                                                                              'Universidad '
                                                                                                                                                                                                                                              'Nacional '
                                                                                                                                                                                                                                              'Autonoma '
                                                                                                                                                                                                                                              'de '
                                                                                                                                                                                                                                              'México, '
                                                                                                                                                                                                                                              'Mexico '
                                                                                                                                                                                                                                              'City, '
                                                                                                                                                                                                                                              'Mexico',
                                                                                                                                                                                                                                              'Departamento '
                                                                                                                                                                                                                                              'de '
                                                                                                                                                                                                                                              'Procesamiento '
                                                                                                                                                                                                                                              'de '
                                                                                                                                                                                                                                              'Señales, '
                                                                                                                                                                                                                                              'Facultad '
                                                                                                                                                                                                                                              'de '
                                                                                                                                                                                                                                              'Ingeniería, '
                                                                                                                                                                                                                                              'Universidad '
                                                                                                                                                                                                                                              'Nacional '
                                                                                                                                                                                                                                              'Autonoma '
                                                                                                                                                                                                                                              'de '
                                                                                                                                                                                                                                              'México, '
                                                                                                                                                                                                                                              'Mexico '
                                                                                                                                                                                                                                              'City, '
                                                                                                                                                                                                                                              'Mexico'),
 'Centro de Estudios en Computación Avanzada, Universidad Nacional Autónoma de México (CECAv-UNAM), Mexico City, Mexico, Centro de Investigación Especializado en el Desarrollo de Tecnologías de la Información y Comunicación, (INFOTEC), Aguascalientes, Mexico': ('Centro '
                                                                                                                                                                                                                                                                      'de '
                                                                                                                                                                                                                                                                      'Estudios '
                                                                                                                                                                                                                                                                      'en '
                                                                                                                                                                                                                                                                      'Computación '
                                                                                                                                                                                                                                                                      'Avanzada, '
                                                                                                                                                                                                                                                                      'Universidad '
                                                                                                                                                                                                                                                                      'Nacional '
                                                                                                                                                                                                                                                                      'Autónoma '
                                                                                                                                                                                                                                                                      'de '
                                                                                                                                                                                                                                                                      'México '
                                                                                                                                                                                                                                                                      '(CECAv-UNAM), '
                                                                                                                                                                                                                                                                      'Mexico '
                                                                                                                                                                                                                                                                      'City, '
                                                                                                                                                                                                                                                                      'Mexico',
                                                                                                                                                                                                                                                                      ''),
 'Centro de Estudios en Computación Avanzada, Universidad Nacional Autónoma de México (CECAv-UNAM), Mexico City, Mexico, Laboratorio Avanzado de Procesamiento de Imágenes, Universidad Nacional Autónoma de México (LaPI-UNAM), Mexico City, Mexico': ('Centro '
                                                                                                                                                                                                                                                        'de '
                                                                                                                                                                                                                                                        'Estudios '
                                                                                                                                                                                                                                                        'en '
                                                                                                                                                                                                                                                        'Computación '
                                                                                                                                                                                                                                                        'Avanzada, '
                                                                                                                                                                                                                                                        'Universidad '
                                                                                                                                                                                                                                                        'Nacional '
                                                                                                                                                                                                                                                        'Autónoma '
                                                                                                                                                                                                                                                        'de '
                                                                                                                                                                                                                                                        'México '
                                                                                                                                                                                                                                                        '(CECAv-UNAM), '
                                                                                                                                                                                                                                                        'Mexico '
                                                                                                                                                                                                                                                        'City, '
                                                                                                                                                                                                                                                        'Mexico',
                                                                                                                                                                                                                                                        'Laboratorio '
                                                                                                                                                                                                                                                        'Avanzado '
                                                                                                                                                                                                                                                        'de '
                                                                                                                                                                                                                                                        'Procesamiento '
                                                                                                                                                                                                                                                        'de '
                                                                                                                                                                                                                                                        'Imágenes, '
                                                                                                                                                                                                                                                        'Universidad '
                                                                                                                                                                                                                                                        'Nacional '
                                                                                                                                                                                                                                                        'Autónoma '
                                                                                                                                                                                                                                                        'de '
                                                                                                                                                                                                                                                        'México '
                                                                                                                                                                                                                                                        '(LaPI-UNAM), '
                                                                                                                                                                                                                                                        'Mexico '
                                                                                                                                                                                                                                                        'City, '
                                                                                                                                                                                                                                                        'Mexico'),
 'Centro de Investigaciones en Geografía Ambiental, Universidad Nacional Autónoma de México, Antigua Carretera a Pátzcuaro 8701, Ex-Hacienda de San José de la Huerta, Morelia 58058, Mexico | Comisión Nacional para el Conocimiento y Uso de la Biodiversidad, Periférico-Insurgentes Sur 4903, Parques del Pedregal, Mexico City 14010, Mexico': ('Centro '
                                                                                                                                                                                                                                                                                                                                                     'de '
                                                                                                                                                                                                                                                                                                                                                     'Investigaciones '
                                                                                                                                                                                                                                                                                                                                                     'en '
                                                                                                                                                                                                                                                                                                                                                     'Geografía '
                                                                                                                                                                                                                                                                                                                                                     'Ambiental, '
                                                                                                                                                                                                                                                                                                                                                     'Universidad '
                                                                                                                                                                                                                                                                                                                                                     'Nacional '
                                                                                                                                                                                                                                                                                                                                                     'Autónoma '
                                                                                                                                                                                                                                                                                                                                                     'de '
                                                                                                                                                                                                                                                                                                                                                     'México, '
                                                                                                                                                                                                                                                                                                                                                     'Antigua '
                                                                                                                                                                                                                                                                                                                                                     'Carretera '
                                                                                                                                                                                                                                                                                                                                                     'a '
                                                                                                                                                                                                                                                                                                                                                     'Pátzcuaro '
                                                                                                                                                                                                                                                                                                                                                     '8701, '
                                                                                                                                                                                                                                                                                                                                                     'Ex-Hacienda '
                                                                                                                                                                                                                                                                                                                                                     'de '
                                                                                                                                                                                                                                                                                                                                                     'San '
                                                                                                                                                                                                                                                                                                                                                     'José '
                                                                                                                                                                                                                                                                                                                                                     'de '
                                                                                                                                                                                                                                                                                                                                                     'la '
                                                                                                                                                                                                                                                                                                                                                     'Huerta, '
                                                                                                                                                                                                                                                                                                                                                     'Morelia '
                                                                                                                                                                                                                                                                                                                                                     '58058, '
                                                                                                                                                                                                                                                                                                                                                     'Mexico',
                                                                                                                                                                                                                                                                                                                                                     ''),
 'Cinvestav Tamaulipas, Center for Research and Advanced Studies, Ciudad Victoria, Mexico, School of Engineering, National Autonomous University of Mexico, Coyoacan, Mexico': ('School '
                                                                                                                                                                                'of '
                                                                                                                                                                                'Engineering, '
                                                                                                                                                                                'National '
                                                                                                                                                                                'Autonomous '
                                                                                                                                                                                'University '
                                                                                                                                                                                'of '
                                                                                                                                                                                'Mexico, '
                                                                                                                                                                                'Coyoacan, '
                                                                                                                                                                                'Mexico',
                                                                                                                                                                                ''),
 'DIFACQUIM Research Group, Department of Pharmacy, School of Chemistry, Universidad Nacional Autónoma de México, Avenida Universidad 3000, México City, 04510, Mexico, Department of Chemistry and Graduate Program in Pharmacology, Center for Research and Advanced Studies of the National Polytechnic Institute, Section 14-740, Mexico City, 07000, Mexico': ('DIFACQUIM '
                                                                                                                                                                                                                                                                                                                                                                    'Research '
                                                                                                                                                                                                                                                                                                                                                                    'Group, '
                                                                                                                                                                                                                                                                                                                                                                    'Department '
                                                                                                                                                                                                                                                                                                                                                                    'of '
                                                                                                                                                                                                                                                                                                                                                                    'Pharmacy, '
                                                                                                                                                                                                                                                                                                                                                                    'School '
                                                                                                                                                                                                                                                                                                                                                                    'of '
                                                                                                                                                                                                                                                                                                                                                                    'Chemistry, '
                                                                                                                                                                                                                                                                                                                                                                    'Universidad '
                                                                                                                                                                                                                                                                                                                                                                    'Nacional '
                                                                                                                                                                                                                                                                                                                                                                    'Autónoma '
                                                                                                                                                                                                                                                                                                                                                                    'de '
                                                                                                                                                                                                                                                                                                                                                                    'México, '
                                                                                                                                                                                                                                                                                                                                                                    'Avenida '
                                                                                                                                                                                                                                                                                                                                                                    'Universidad '
                                                                                                                                                                                                                                                                                                                                                                    '3000, '
                                                                                                                                                                                                                                                                                                                                                                    'México '
                                                                                                                                                                                                                                                                                                                                                                    'City, '
                                                                                                                                                                                                                                                                                                                                                                    '04510, '
                                                                                                                                                                                                                                                                                                                                                                    'Mexico',
                                                                                                                                                                                                                                                                                                                                                                    ''),
 'Departamento de Genómica Computacional, Instituto Nacional de Medicina Genómica, Ciudad de México, CDMX 14610, Mexico, Centro de Ciencias de la Complejidad, Universidad Nacional Autónoma de México, Ciudad de México, CDMX 04510, Mexico': ('Centro '
                                                                                                                                                                                                                                                'de '
                                                                                                                                                                                                                                                'Ciencias '
                                                                                                                                                                                                                                                'de '
                                                                                                                                                                                                                                                'la '
                                                                                                                                                                                                                                                'Complejidad, '
                                                                                                                                                                                                                                                'Universidad '
                                                                                                                                                                                                                                                'Nacional '
                                                                                                                                                                                                                                                'Autónoma '
                                                                                                                                                                                                                                                'de '
                                                                                                                                                                                                                                                'México, '
                                                                                                                                                                                                                                                'Ciudad '
                                                                                                                                                                                                                                                'de '
                                                                                                                                                                                                                                                'México, '
                                                                                                                                                                                                                                                'CDMX '
                                                                                                                                                                                                                                                '04510, '
                                                                                                                                                                                                                                                'Mexico',
                                                                                                                                                                                                                                                ''),
 'Departamento de Genómica Computacional, Instituto Nacional de Medicina Genómica, Ciudad de México, CDMX 14610, Mexico, Consejo Nacional de Ciencia y Tecnologia, IxM CONAHCYT, Ciudad de México, CDMX 03940, Mexico, Centro de Ciencias de la Complejidad, Universidad Nacional Autónoma de México, Ciudad de México, CDMX 04510, Mexico': ('Centro '
                                                                                                                                                                                                                                                                                                                                              'de '
                                                                                                                                                                                                                                                                                                                                              'Ciencias '
                                                                                                                                                                                                                                                                                                                                              'de '
                                                                                                                                                                                                                                                                                                                                              'la '
                                                                                                                                                                                                                                                                                                                                              'Complejidad, '
                                                                                                                                                                                                                                                                                                                                              'Universidad '
                                                                                                                                                                                                                                                                                                                                              'Nacional '
                                                                                                                                                                                                                                                                                                                                              'Autónoma '
                                                                                                                                                                                                                                                                                                                                              'de '
                                                                                                                                                                                                                                                                                                                                              'México, '
                                                                                                                                                                                                                                                                                                                                              'Ciudad '
                                                                                                                                                                                                                                                                                                                                              'de '
                                                                                                                                                                                                                                                                                                                                              'México, '
                                                                                                                                                                                                                                                                                                                                              'CDMX '
                                                                                                                                                                                                                                                                                                                                              '04510, '
                                                                                                                                                                                                                                                                                                                                              'Mexico',
                                                                                                                                                                                                                                                                                                                                              ''),
 'Departamento de Procesamiento de Señales, Facultad de Ingeniería, Universidad Nacional Autónoma de México, Mexico City, Mexico | Centro de Estudios en Computación Avanzada, Universidad Nacional Autónoma de México, Mexico City, Mexico': ('Departamento '
                                                                                                                                                                                                                                               'de '
                                                                                                                                                                                                                                               'Procesamiento '
                                                                                                                                                                                                                                               'de '
                                                                                                                                                                                                                                               'Señales, '
                                                                                                                                                                                                                                               'Facultad '
                                                                                                                                                                                                                                               'de '
                                                                                                                                                                                                                                               'Ingeniería, '
                                                                                                                                                                                                                                               'Universidad '
                                                                                                                                                                                                                                               'Nacional '
                                                                                                                                                                                                                                               'Autónoma '
                                                                                                                                                                                                                                               'de '
                                                                                                                                                                                                                                               'México, '
                                                                                                                                                                                                                                               'Mexico '
                                                                                                                                                                                                                                               'City, '
                                                                                                                                                                                                                                               'Mexico',
                                                                                                                                                                                                                                               'Centro '
                                                                                                                                                                                                                                               'de '
                                                                                                                                                                                                                                               'Estudios '
                                                                                                                                                                                                                                               'en '
                                                                                                                                                                                                                                               'Computación '
                                                                                                                                                                                                                                               'Avanzada, '
                                                                                                                                                                                                                                               'Universidad '
                                                                                                                                                                                                                                               'Nacional '
                                                                                                                                                                                                                                               'Autónoma '
                                                                                                                                                                                                                                               'de '
                                                                                                                                                                                                                                               'México, '
                                                                                                                                                                                                                                               'Mexico '
                                                                                                                                                                                                                                               'City, '
                                                                                                                                                                                                                                               'Mexico'),
 'Departamento de Procesamiento de Señales, Facultad de Ingeniería, Universidad Nacional Autónoma de México, Mexico City, Mexico, Centro de Estudios en Computación Avanzada, Universidad Nacional Autónoma de México, Mexico City, Mexico': ('Departamento '
                                                                                                                                                                                                                                              'de '
                                                                                                                                                                                                                                              'Procesamiento '
                                                                                                                                                                                                                                              'de '
                                                                                                                                                                                                                                              'Señales, '
                                                                                                                                                                                                                                              'Facultad '
                                                                                                                                                                                                                                              'de '
                                                                                                                                                                                                                                              'Ingeniería, '
                                                                                                                                                                                                                                              'Universidad '
                                                                                                                                                                                                                                              'Nacional '
                                                                                                                                                                                                                                              'Autónoma '
                                                                                                                                                                                                                                              'de '
                                                                                                                                                                                                                                              'México, '
                                                                                                                                                                                                                                              'Mexico '
                                                                                                                                                                                                                                              'City, '
                                                                                                                                                                                                                                              'Mexico',
                                                                                                                                                                                                                                              'Centro '
                                                                                                                                                                                                                                              'de '
                                                                                                                                                                                                                                              'Estudios '
                                                                                                                                                                                                                                              'en '
                                                                                                                                                                                                                                              'Computación '
                                                                                                                                                                                                                                              'Avanzada, '
                                                                                                                                                                                                                                              'Universidad '
                                                                                                                                                                                                                                              'Nacional '
                                                                                                                                                                                                                                              'Autónoma '
                                                                                                                                                                                                                                              'de '
                                                                                                                                                                                                                                              'México, '
                                                                                                                                                                                                                                              'Mexico '
                                                                                                                                                                                                                                              'City, '
                                                                                                                                                                                                                                              'Mexico'),
 'Departamento de Zoología, Instituto de Biología, Universidad Nacional Autónoma de México, Ciudad de México, 04510, Mexico, Fondo Mexicano para la Conservación de la Naturaleza A.C., Ciudad de México, 03900, Mexico, Centro de Investigación en Ciencias de Información Geoespacial, Ciudad de México, 14240, Mexico': ('Departamento '
                                                                                                                                                                                                                                                                                                                            'de '
                                                                                                                                                                                                                                                                                                                            'Zoología, '
                                                                                                                                                                                                                                                                                                                            'Instituto '
                                                                                                                                                                                                                                                                                                                            'de '
                                                                                                                                                                                                                                                                                                                            'Biología, '
                                                                                                                                                                                                                                                                                                                            'Universidad '
                                                                                                                                                                                                                                                                                                                            'Nacional '
                                                                                                                                                                                                                                                                                                                            'Autónoma '
                                                                                                                                                                                                                                                                                                                            'de '
                                                                                                                                                                                                                                                                                                                            'México, '
                                                                                                                                                                                                                                                                                                                            'Ciudad '
                                                                                                                                                                                                                                                                                                                            'de '
                                                                                                                                                                                                                                                                                                                            'México, '
                                                                                                                                                                                                                                                                                                                            '04510, '
                                                                                                                                                                                                                                                                                                                            'Mexico',
                                                                                                                                                                                                                                                                                                                            ''),
 'Department of Chemical, Industrial and Food Engineering, Universidad Iberoamericana, Mexico City, 01219, Mexico, Facultad de Ingeniería, Universidad Nacional Autónoma de México, Mexico City, 04510, Mexico': ('Facultad '
                                                                                                                                                                                                                  'de '
                                                                                                                                                                                                                  'Ingeniería, '
                                                                                                                                                                                                                  'Universidad '
                                                                                                                                                                                                                  'Nacional '
                                                                                                                                                                                                                  'Autónoma '
                                                                                                                                                                                                                  'de '
                                                                                                                                                                                                                  'México, '
                                                                                                                                                                                                                  'Mexico '
                                                                                                                                                                                                                  'City, '
                                                                                                                                                                                                                  '04510, '
                                                                                                                                                                                                                  'Mexico',
                                                                                                                                                                                                                  ''),
 'Department of Mathematics, Faculty of Sciences, UNAM, Mexico City, Mexico, Department of Probability and Statistics, IIMAS, UNAM, Mexico City, Mexico': ('Department '
                                                                                                                                                           'of '
                                                                                                                                                           'Mathematics, '
                                                                                                                                                           'Faculty '
                                                                                                                                                           'of '
                                                                                                                                                           'Sciences, '
                                                                                                                                                           'UNAM, '
                                                                                                                                                           'Mexico '
                                                                                                                                                           'City, '
                                                                                                                                                           'Mexico',
                                                                                                                                                           'Department '
                                                                                                                                                           'of '
                                                                                                                                                           'Probability '
                                                                                                                                                           'and '
                                                                                                                                                           'Statistics, '
                                                                                                                                                           'IIMAS, '
                                                                                                                                                           'UNAM, '
                                                                                                                                                           'Mexico '
                                                                                                                                                           'City, '
                                                                                                                                                           'Mexico'),
 'Division of Infectious Diseases, Department of Medicine, Weill Cornell Medicine, New York, 10021, NY, United States, Programa de Doctorado en Ciencias Biomédicas, Universidad Nacional Autónoma de México, Ciudad de México, CDMX 04510, Mexico, Departamento de Genómica Computacional, Instituto Nacional de Medicina Genómica, Ciudad de México, CDMX 14610, Mexico, Institute of Translational Research, Feinstein Institutes for Medical Research, Northwell Health, Manhasset, 11030, NY, United States': ('Programa '
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    'de '
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    'Doctorado '
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    'en '
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    'Ciencias '
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    'Biomédicas, '
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    'Universidad '
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    'Nacional '
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    'Autónoma '
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    'de '
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    'México, '
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    'Ciudad '
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    'de '
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    'México, '
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    'CDMX '
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    '04510, '
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    'Mexico',
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    ''),
 'Facultad de Ciencias, Universidad Nacional Autónoma de México, Mexico City, 04510, Mexico, Centro de Ciencias de la Complejidad, Universidad Nacional Autónoma de México, Mexico City, 04510, Mexico': ('Facultad '
                                                                                                                                                                                                          'de '
                                                                                                                                                                                                          'Ciencias, '
                                                                                                                                                                                                          'Universidad '
                                                                                                                                                                                                          'Nacional '
                                                                                                                                                                                                          'Autónoma '
                                                                                                                                                                                                          'de '
                                                                                                                                                                                                          'México, '
                                                                                                                                                                                                          'Mexico '
                                                                                                                                                                                                          'City, '
                                                                                                                                                                                                          '04510, '
                                                                                                                                                                                                          'Mexico',
                                                                                                                                                                                                          'Centro '
                                                                                                                                                                                                          'de '
                                                                                                                                                                                                          'Ciencias '
                                                                                                                                                                                                          'de '
                                                                                                                                                                                                          'la '
                                                                                                                                                                                                          'Complejidad, '
                                                                                                                                                                                                          'Universidad '
                                                                                                                                                                                                          'Nacional '
                                                                                                                                                                                                          'Autónoma '
                                                                                                                                                                                                          'de '
                                                                                                                                                                                                          'México, '
                                                                                                                                                                                                          'Mexico '
                                                                                                                                                                                                          'City, '
                                                                                                                                                                                                          '04510, '
                                                                                                                                                                                                          'Mexico'),
 'Facultad de Ciencias, Universidad Nacional Autónoma de México, Mexico City, 04510, Mexico, Centro de Ciencias de la Complejidad, Universidad Nacional Autónoma de México, Mexico City, 04510, Mexico, Instituto de Fisica Interdisciplinar y Sistemas Complejos, Universidad de las Islas Baleares, Palma de Mallorca, 07122, Spain': ('Facultad '
                                                                                                                                                                                                                                                                                                                                         'de '
                                                                                                                                                                                                                                                                                                                                         'Ciencias, '
                                                                                                                                                                                                                                                                                                                                         'Universidad '
                                                                                                                                                                                                                                                                                                                                         'Nacional '
                                                                                                                                                                                                                                                                                                                                         'Autónoma '
                                                                                                                                                                                                                                                                                                                                         'de '
                                                                                                                                                                                                                                                                                                                                         'México, '
                                                                                                                                                                                                                                                                                                                                         'Mexico '
                                                                                                                                                                                                                                                                                                                                         'City, '
                                                                                                                                                                                                                                                                                                                                         '04510, '
                                                                                                                                                                                                                                                                                                                                         'Mexico',
                                                                                                                                                                                                                                                                                                                                         'Centro '
                                                                                                                                                                                                                                                                                                                                         'de '
                                                                                                                                                                                                                                                                                                                                         'Ciencias '
                                                                                                                                                                                                                                                                                                                                         'de '
                                                                                                                                                                                                                                                                                                                                         'la '
                                                                                                                                                                                                                                                                                                                                         'Complejidad, '
                                                                                                                                                                                                                                                                                                                                         'Universidad '
                                                                                                                                                                                                                                                                                                                                         'Nacional '
                                                                                                                                                                                                                                                                                                                                         'Autónoma '
                                                                                                                                                                                                                                                                                                                                         'de '
                                                                                                                                                                                                                                                                                                                                         'México, '
                                                                                                                                                                                                                                                                                                                                         'Mexico '
                                                                                                                                                                                                                                                                                                                                         'City, '
                                                                                                                                                                                                                                                                                                                                         '04510, '
                                                                                                                                                                                                                                                                                                                                         'Mexico'),
 'Facultad de Ciencias, Universidad Nacional Autónoma de México, Mexico City, Mexico, Consejo Nacional de Ciencia y Tecnología, Mexico City, Mexico, Centro Nacional de Investigación y Desarrollo Tecnológico - Tecnológico Nacional de México, Mexico City, Mexico': ('Facultad '
                                                                                                                                                                                                                                                                        'de '
                                                                                                                                                                                                                                                                        'Ciencias, '
                                                                                                                                                                                                                                                                        'Universidad '
                                                                                                                                                                                                                                                                        'Nacional '
                                                                                                                                                                                                                                                                        'Autónoma '
                                                                                                                                                                                                                                                                        'de '
                                                                                                                                                                                                                                                                        'México, '
                                                                                                                                                                                                                                                                        'Mexico '
                                                                                                                                                                                                                                                                        'City, '
                                                                                                                                                                                                                                                                        'Mexico',
                                                                                                                                                                                                                                                                        ''),
 'Facultad de Ingeniería, UNAM, Av. Universidad 3000, Ciudad Universitaria, Coyoacán, Mexico City, 04510, Mexico, Centro de estudios en computación avanzada, UNAM, Av. Universidad 3000, Ciudad Universitaria, Coyoacán, Mexico City, 04510, Mexico': ('Facultad '
                                                                                                                                                                                                                                                        'de '
                                                                                                                                                                                                                                                        'Ingeniería, '
                                                                                                                                                                                                                                                        'UNAM, '
                                                                                                                                                                                                                                                        'Av. '
                                                                                                                                                                                                                                                        'Universidad '
                                                                                                                                                                                                                                                        '3000, '
                                                                                                                                                                                                                                                        'Ciudad '
                                                                                                                                                                                                                                                        'Universitaria, '
                                                                                                                                                                                                                                                        'Coyoacán, '
                                                                                                                                                                                                                                                        'Mexico '
                                                                                                                                                                                                                                                        'City, '
                                                                                                                                                                                                                                                        '04510, '
                                                                                                                                                                                                                                                        'Mexico',
                                                                                                                                                                                                                                                        'Centro '
                                                                                                                                                                                                                                                        'de '
                                                                                                                                                                                                                                                        'estudios '
                                                                                                                                                                                                                                                        'en '
                                                                                                                                                                                                                                                        'computación '
                                                                                                                                                                                                                                                        'avanzada, '
                                                                                                                                                                                                                                                        'UNAM, '
                                                                                                                                                                                                                                                        'Av. '
                                                                                                                                                                                                                                                        'Universidad '
                                                                                                                                                                                                                                                        '3000, '
                                                                                                                                                                                                                                                        'Ciudad '
                                                                                                                                                                                                                                                        'Universitaria, '
                                                                                                                                                                                                                                                        'Coyoacán, '
                                                                                                                                                                                                                                                        'Mexico '
                                                                                                                                                                                                                                                        'City, '
                                                                                                                                                                                                                                                        '04510, '
                                                                                                                                                                                                                                                        'Mexico'),
 'Facultad de Ingeniería, Universidad Nacional Autónoma de Mexico, Av. Universidad 3000, Coyoacan, Mexico City, 04510, Mexico, Centro de Estudios en Computación Avanzada, Universidad Nacional Autónoma de Mexico, Av. Universidad 3000, Coyoacan, Mexico City, 04510, Mexico': ('Facultad '
                                                                                                                                                                                                                                                                                  'de '
                                                                                                                                                                                                                                                                                  'Ingeniería, '
                                                                                                                                                                                                                                                                                  'Universidad '
                                                                                                                                                                                                                                                                                  'Nacional '
                                                                                                                                                                                                                                                                                  'Autónoma '
                                                                                                                                                                                                                                                                                  'de '
                                                                                                                                                                                                                                                                                  'Mexico, '
                                                                                                                                                                                                                                                                                  'Av. '
                                                                                                                                                                                                                                                                                  'Universidad '
                                                                                                                                                                                                                                                                                  '3000, '
                                                                                                                                                                                                                                                                                  'Coyoacan, '
                                                                                                                                                                                                                                                                                  'Mexico '
                                                                                                                                                                                                                                                                                  'City, '
                                                                                                                                                                                                                                                                                  '04510, '
                                                                                                                                                                                                                                                                                  'Mexico',
                                                                                                                                                                                                                                                                                  'Centro '
                                                                                                                                                                                                                                                                                  'de '
                                                                                                                                                                                                                                                                                  'Estudios '
                                                                                                                                                                                                                                                                                  'en '
                                                                                                                                                                                                                                                                                  'Computación '
                                                                                                                                                                                                                                                                                  'Avanzada, '
                                                                                                                                                                                                                                                                                  'Universidad '
                                                                                                                                                                                                                                                                                  'Nacional '
                                                                                                                                                                                                                                                                                  'Autónoma '
                                                                                                                                                                                                                                                                                  'de '
                                                                                                                                                                                                                                                                                  'Mexico, '
                                                                                                                                                                                                                                                                                  'Av. '
                                                                                                                                                                                                                                                                                  'Universidad '
                                                                                                                                                                                                                                                                                  '3000, '
                                                                                                                                                                                                                                                                                  'Coyoacan, '
                                                                                                                                                                                                                                                                                  'Mexico '
                                                                                                                                                                                                                                                                                  'City, '
                                                                                                                                                                                                                                                                                  '04510, '
                                                                                                                                                                                                                                                                                  'Mexico'),
 'Facultad de Ingeniería, Universidad Nacional Autónoma de México (LaPI-UNAM), Mexico City, Mexico, Centro de Investigación Especializado en el Desarrollo de Tecnologías de la Información y Comunicación, (INFOTEC), Aguascalientes, Mexico': ('Facultad '
                                                                                                                                                                                                                                                 'de '
                                                                                                                                                                                                                                                 'Ingeniería, '
                                                                                                                                                                                                                                                 'Universidad '
                                                                                                                                                                                                                                                 'Nacional '
                                                                                                                                                                                                                                                 'Autónoma '
                                                                                                                                                                                                                                                 'de '
                                                                                                                                                                                                                                                 'México '
                                                                                                                                                                                                                                                 '(LaPI-UNAM), '
                                                                                                                                                                                                                                                 'Mexico '
                                                                                                                                                                                                                                                 'City, '
                                                                                                                                                                                                                                                 'Mexico',
                                                                                                                                                                                                                                                 ''),
 'Facultad de Ingeniería, Universidad Nacional Autónoma de México, Ciudad de México, Mexico, Centro de Estudios en Computación Avanzada, Universidad Nacional Autónoma de México, Mexico': ('Facultad '
                                                                                                                                                                                            'de '
                                                                                                                                                                                            'Ingeniería, '
                                                                                                                                                                                            'Universidad '
                                                                                                                                                                                            'Nacional '
                                                                                                                                                                                            'Autónoma '
                                                                                                                                                                                            'de '
                                                                                                                                                                                            'México, '
                                                                                                                                                                                            'Ciudad '
                                                                                                                                                                                            'de '
                                                                                                                                                                                            'México, '
                                                                                                                                                                                            'Mexico',
                                                                                                                                                                                            'Centro '
                                                                                                                                                                                            'de '
                                                                                                                                                                                            'Estudios '
                                                                                                                                                                                            'en '
                                                                                                                                                                                            'Computación '
                                                                                                                                                                                            'Avanzada, '
                                                                                                                                                                                            'Universidad '
                                                                                                                                                                                            'Nacional '
                                                                                                                                                                                            'Autónoma '
                                                                                                                                                                                            'de '
                                                                                                                                                                                            'México, '
                                                                                                                                                                                            'Mexico'),
 'Facultad de Ingeniería, Universidad Nacional Autónoma de México, Mexico City, Mexico | Centro de Estudios en Computación Avanzada, Universidad Nacional Autónoma de México, Mexico City, Mexico': ('Facultad '
                                                                                                                                                                                                     'de '
                                                                                                                                                                                                     'Ingeniería, '
                                                                                                                                                                                                     'Universidad '
                                                                                                                                                                                                     'Nacional '
                                                                                                                                                                                                     'Autónoma '
                                                                                                                                                                                                     'de '
                                                                                                                                                                                                     'México, '
                                                                                                                                                                                                     'Mexico '
                                                                                                                                                                                                     'City, '
                                                                                                                                                                                                     'Mexico',
                                                                                                                                                                                                     'Centro '
                                                                                                                                                                                                     'de '
                                                                                                                                                                                                     'Estudios '
                                                                                                                                                                                                     'en '
                                                                                                                                                                                                     'Computación '
                                                                                                                                                                                                     'Avanzada, '
                                                                                                                                                                                                     'Universidad '
                                                                                                                                                                                                     'Nacional '
                                                                                                                                                                                                     'Autónoma '
                                                                                                                                                                                                     'de '
                                                                                                                                                                                                     'México, '
                                                                                                                                                                                                     'Mexico '
                                                                                                                                                                                                     'City, '
                                                                                                                                                                                                     'Mexico'),
 'Facultad de Ingeniería, Universidad Nacional Autónoma de México, Mexico City, Mexico, Centro de Estudios en Computación Avanzada, Universidad Nacional Autónoma de México, Mexico City, Mexico': ('Facultad '
                                                                                                                                                                                                    'de '
                                                                                                                                                                                                    'Ingeniería, '
                                                                                                                                                                                                    'Universidad '
                                                                                                                                                                                                    'Nacional '
                                                                                                                                                                                                    'Autónoma '
                                                                                                                                                                                                    'de '
                                                                                                                                                                                                    'México, '
                                                                                                                                                                                                    'Mexico '
                                                                                                                                                                                                    'City, '
                                                                                                                                                                                                    'Mexico',
                                                                                                                                                                                                    'Centro '
                                                                                                                                                                                                    'de '
                                                                                                                                                                                                    'Estudios '
                                                                                                                                                                                                    'en '
                                                                                                                                                                                                    'Computación '
                                                                                                                                                                                                    'Avanzada, '
                                                                                                                                                                                                    'Universidad '
                                                                                                                                                                                                    'Nacional '
                                                                                                                                                                                                    'Autónoma '
                                                                                                                                                                                                    'de '
                                                                                                                                                                                                    'México, '
                                                                                                                                                                                                    'Mexico '
                                                                                                                                                                                                    'City, '
                                                                                                                                                                                                    'Mexico'),
 'Grupo de Ingeniería Lingüística - UNAM, Mexico, Departament de Filologia Catalana i Lingüística General, Universitat de Barcelona, Spain': ('Grupo '
                                                                                                                                              'de '
                                                                                                                                              'Ingeniería '
                                                                                                                                              'Lingüística '
                                                                                                                                              '- '
                                                                                                                                              'UNAM, '
                                                                                                                                              'Mexico',
                                                                                                                                              ''),
 'Grupo de Ingeniería Lingüística - UNAM, Mexico, Posgrado en Ciencias e Ingeniería de la Computación - UNAM, Mexico': ('Grupo '
                                                                                                                        'de '
                                                                                                                        'Ingeniería '
                                                                                                                        'Lingüística '
                                                                                                                        '- '
                                                                                                                        'UNAM, '
                                                                                                                        'Mexico',
                                                                                                                        'Posgrado '
                                                                                                                        'en '
                                                                                                                        'Ciencias '
                                                                                                                        'e '
                                                                                                                        'Ingeniería '
                                                                                                                        'de '
                                                                                                                        'la '
                                                                                                                        'Computación '
                                                                                                                        '- '
                                                                                                                        'UNAM, '
                                                                                                                        'Mexico'),
 'Grupo de Ingeniería Lingüística, UNAM, Mexico | Departament de Filologia Catalana i Lingüística General, Universitat de Barcelona, Spain': ('Grupo '
                                                                                                                                              'de '
                                                                                                                                              'Ingeniería '
                                                                                                                                              'Lingüística, '
                                                                                                                                              'UNAM, '
                                                                                                                                              'Mexico',
                                                                                                                                              ''),
 'Grupo de Ingeniería Lingüística, UNAM, Mexico | UNAM, Mexico': ('Grupo de Ingeniería Lingüística, UNAM, Mexico',
                                                                  'UNAM, Mexico'),
 'Grupo de Ingeniería Lingüística, UNAM, Mexico, Departament de Filologia Catalana i Lingüística General, Universitat de Barcelona, Spain': ('Grupo '
                                                                                                                                             'de '
                                                                                                                                             'Ingeniería '
                                                                                                                                             'Lingüística, '
                                                                                                                                             'UNAM, '
                                                                                                                                             'Mexico',
                                                                                                                                             ''),
 'IRIF, University Paris Cité, CNRS, Paris, France | Instituto de Matemáticas, UNAM, Mexico City, Mexico': ('Instituto '
                                                                                                            'de '
                                                                                                            'Matemáticas, '
                                                                                                            'UNAM, '
                                                                                                            'Mexico '
                                                                                                            'City, '
                                                                                                            'Mexico',
                                                                                                            ''),
 'IRIF, Université Paris Cité and CNRS, Paris, France, Instituto de Matematicas, Universidad Nacional Autónoma de México, Mexico City, Mexico': ('Instituto '
                                                                                                                                                 'de '
                                                                                                                                                 'Matematicas, '
                                                                                                                                                 'Universidad '
                                                                                                                                                 'Nacional '
                                                                                                                                                 'Autónoma '
                                                                                                                                                 'de '
                                                                                                                                                 'México, '
                                                                                                                                                 'Mexico '
                                                                                                                                                 'City, '
                                                                                                                                                 'Mexico',
                                                                                                                                                 ''),
 'Institute of Advanced Materials for Sustainable Manufacturing, Tecnologico de Monterrey, N.L, Monterrey; 64849, Mexico | Universidad Nacional Autonoma de Mexico, Coyoacan CDMX; 04510, Mexico | Instituto Nacional de Enfermedades Respiratorias Ismael Cosio Villegas, Tlalpan CDMX; 14080, Mexico': ('Universidad '
                                                                                                                                                                                                                                                                                                          'Nacional '
                                                                                                                                                                                                                                                                                                          'Autonoma '
                                                                                                                                                                                                                                                                                                          'de '
                                                                                                                                                                                                                                                                                                          'Mexico, '
                                                                                                                                                                                                                                                                                                          'Coyoacan '
                                                                                                                                                                                                                                                                                                          'CDMX; '
                                                                                                                                                                                                                                                                                                          '04510, '
                                                                                                                                                                                                                                                                                                          'Mexico',
                                                                                                                                                                                                                                                                                                          ''),
 'Instituto de Biotecnología, Universidad Nacional Autónoma de México, Morelos, Cuernavaca, Mexico, Posgrado en Ciencia e Ingeniería de la Computación, Universidad Nacional Autónoma de México, Mexico City, Mexico': ('Instituto '
                                                                                                                                                                                                                        'de '
                                                                                                                                                                                                                        'Biotecnología, '
                                                                                                                                                                                                                        'Universidad '
                                                                                                                                                                                                                        'Nacional '
                                                                                                                                                                                                                        'Autónoma '
                                                                                                                                                                                                                        'de '
                                                                                                                                                                                                                        'México, '
                                                                                                                                                                                                                        'Morelos, '
                                                                                                                                                                                                                        'Cuernavaca, '
                                                                                                                                                                                                                        'Mexico',
                                                                                                                                                                                                                        'Posgrado '
                                                                                                                                                                                                                        'en '
                                                                                                                                                                                                                        'Ciencia '
                                                                                                                                                                                                                        'e '
                                                                                                                                                                                                                        'Ingeniería '
                                                                                                                                                                                                                        'de '
                                                                                                                                                                                                                        'la '
                                                                                                                                                                                                                        'Computación, '
                                                                                                                                                                                                                        'Universidad '
                                                                                                                                                                                                                        'Nacional '
                                                                                                                                                                                                                        'Autónoma '
                                                                                                                                                                                                                        'de '
                                                                                                                                                                                                                        'México, '
                                                                                                                                                                                                                        'Mexico '
                                                                                                                                                                                                                        'City, '
                                                                                                                                                                                                                        'Mexico'),
 'Instituto de Ciencias Nucleares, Universidad Nacional Autónoma de México, Apratado Postal 70-543, Cd. Mx., 04510, Mexico, Institute of Neuroscience and Medicine (INM-1), Research Centre Jülich, Jülich, D-52425, Germany': ('Instituto '
                                                                                                                                                                                                                                'de '
                                                                                                                                                                                                                                'Ciencias '
                                                                                                                                                                                                                                'Nucleares, '
                                                                                                                                                                                                                                'Universidad '
                                                                                                                                                                                                                                'Nacional '
                                                                                                                                                                                                                                'Autónoma '
                                                                                                                                                                                                                                'de '
                                                                                                                                                                                                                                'México, '
                                                                                                                                                                                                                                'Apratado '
                                                                                                                                                                                                                                'Postal '
                                                                                                                                                                                                                                '70-543, '
                                                                                                                                                                                                                                'Cd. '
                                                                                                                                                                                                                                'Mx., '
                                                                                                                                                                                                                                '04510, '
                                                                                                                                                                                                                                'Mexico',
                                                                                                                                                                                                                                ''),
 'Instituto de Fisiología Celular - Neurociencias and Centro de Ciencias de la Complejidad, Universidad Nacional Autónoma de México, México': ('Instituto '
                                                                                                                                               'de '
                                                                                                                                               'Fisiología '
                                                                                                                                               'Celular '
                                                                                                                                               '- '
                                                                                                                                               'Neurociencias, '
                                                                                                                                               'Universidad '
                                                                                                                                               'Nacional '
                                                                                                                                               'Autónoma '
                                                                                                                                               'de '
                                                                                                                                               'México, '
                                                                                                                                               'México',
                                                                                                                                               'Centro '
                                                                                                                                               'de '
                                                                                                                                               'Ciencias '
                                                                                                                                               'de '
                                                                                                                                               'la '
                                                                                                                                               'Complejidad, '
                                                                                                                                               'Universidad '
                                                                                                                                               'Nacional '
                                                                                                                                               'Autónoma '
                                                                                                                                               'de '
                                                                                                                                               'México, '
                                                                                                                                               'México'),
 'Instituto de Física Universidad Nacional Autónoma de México, México, C.P. 04510, Mexico, BASLEARN, BASF-TU joint Lab, Technische Universität Berlin, Berlin, 10587, Germany': ('Instituto '
                                                                                                                                                                                 'de '
                                                                                                                                                                                 'Física '
                                                                                                                                                                                 'Universidad '
                                                                                                                                                                                 'Nacional '
                                                                                                                                                                                 'Autónoma '
                                                                                                                                                                                 'de '
                                                                                                                                                                                 'México, '
                                                                                                                                                                                 'México, '
                                                                                                                                                                                 'C.P. '
                                                                                                                                                                                 '04510, '
                                                                                                                                                                                 'Mexico',
                                                                                                                                                                                 ''),
 'Instituto de Ingeniería, Universidad Nacional Autónoma de México, Ciudad de México, 04510, Mexico, Departament de Filologia Catalana i Lingüística General, Universitat de Barcelona, Barcelona, Spain': ('Instituto '
                                                                                                                                                                                                            'de '
                                                                                                                                                                                                            'Ingeniería, '
                                                                                                                                                                                                            'Universidad '
                                                                                                                                                                                                            'Nacional '
                                                                                                                                                                                                            'Autónoma '
                                                                                                                                                                                                            'de '
                                                                                                                                                                                                            'México, '
                                                                                                                                                                                                            'Ciudad '
                                                                                                                                                                                                            'de '
                                                                                                                                                                                                            'México, '
                                                                                                                                                                                                            '04510, '
                                                                                                                                                                                                            'Mexico',
                                                                                                                                                                                                            ''),
 'Instituto de Ingeniería, Universidad Nacional Autónoma de México, Ciudad de México; 04510, Mexico | Departament de Filologia Catalana i Lingüística General, Universitat de Barcelona, Barcelona, Spain': ('Instituto '
                                                                                                                                                                                                             'de '
                                                                                                                                                                                                             'Ingeniería, '
                                                                                                                                                                                                             'Universidad '
                                                                                                                                                                                                             'Nacional '
                                                                                                                                                                                                             'Autónoma '
                                                                                                                                                                                                             'de '
                                                                                                                                                                                                             'México, '
                                                                                                                                                                                                             'Ciudad '
                                                                                                                                                                                                             'de '
                                                                                                                                                                                                             'México; '
                                                                                                                                                                                                             '04510, '
                                                                                                                                                                                                             'Mexico',
                                                                                                                                                                                                             ''),
 'Instituto de Ingeniería, Universidad Nacional Autónoma de México, Mexico, Facultat de Filologia i Comunicació, Universitat de Barcelona, Spain': ('Instituto '
                                                                                                                                                    'de '
                                                                                                                                                    'Ingeniería, '
                                                                                                                                                    'Universidad '
                                                                                                                                                    'Nacional '
                                                                                                                                                    'Autónoma '
                                                                                                                                                    'de '
                                                                                                                                                    'México, '
                                                                                                                                                    'Mexico',
                                                                                                                                                    ''),
 'Instituto de Ingeniería, Universidad Nacional Autónoma de México, Mexico, Insituto de Investigaciones Bibliográficas, Universidad Nacional Autónoma de México, Mexico': ('Instituto '
                                                                                                                                                                           'de '
                                                                                                                                                                           'Ingeniería, '
                                                                                                                                                                           'Universidad '
                                                                                                                                                                           'Nacional '
                                                                                                                                                                           'Autónoma '
                                                                                                                                                                           'de '
                                                                                                                                                                           'México, '
                                                                                                                                                                           'Mexico',
                                                                                                                                                                           'Insituto '
                                                                                                                                                                           'de '
                                                                                                                                                                           'Investigaciones '
                                                                                                                                                                           'Bibliográficas, '
                                                                                                                                                                           'Universidad '
                                                                                                                                                                           'Nacional '
                                                                                                                                                                           'Autónoma '
                                                                                                                                                                           'de '
                                                                                                                                                                           'México, '
                                                                                                                                                                           'Mexico'),
 'Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas, Universidad Nacional Autónoma de México, Ciudad Universitaria, CDMX, 04510, Mexico, Facultad de Ciencias, Universidad Nacional Autónoma de México, Ciudad Universitaria, CDMX, 04510, Mexico': ('Instituto '
                                                                                                                                                                                                                                                                       'de '
                                                                                                                                                                                                                                                                       'Investigaciones '
                                                                                                                                                                                                                                                                       'en '
                                                                                                                                                                                                                                                                       'Matemáticas '
                                                                                                                                                                                                                                                                       'Aplicadas '
                                                                                                                                                                                                                                                                       'y '
                                                                                                                                                                                                                                                                       'en '
                                                                                                                                                                                                                                                                       'Sistemas, '
                                                                                                                                                                                                                                                                       'Universidad '
                                                                                                                                                                                                                                                                       'Nacional '
                                                                                                                                                                                                                                                                       'Autónoma '
                                                                                                                                                                                                                                                                       'de '
                                                                                                                                                                                                                                                                       'México, '
                                                                                                                                                                                                                                                                       'Ciudad '
                                                                                                                                                                                                                                                                       'Universitaria, '
                                                                                                                                                                                                                                                                       'CDMX, '
                                                                                                                                                                                                                                                                       '04510, '
                                                                                                                                                                                                                                                                       'Mexico',
                                                                                                                                                                                                                                                                       'Facultad '
                                                                                                                                                                                                                                                                       'de '
                                                                                                                                                                                                                                                                       'Ciencias, '
                                                                                                                                                                                                                                                                       'Universidad '
                                                                                                                                                                                                                                                                       'Nacional '
                                                                                                                                                                                                                                                                       'Autónoma '
                                                                                                                                                                                                                                                                       'de '
                                                                                                                                                                                                                                                                       'México, '
                                                                                                                                                                                                                                                                       'Ciudad '
                                                                                                                                                                                                                                                                       'Universitaria, '
                                                                                                                                                                                                                                                                       'CDMX, '
                                                                                                                                                                                                                                                                       '04510, '
                                                                                                                                                                                                                                                                       'Mexico'),
 'Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas-UNAM, Mexico, Universidad Modelo, Mérida, Mexico': ('Instituto '
                                                                                                                          'de '
                                                                                                                          'Investigaciones '
                                                                                                                          'en '
                                                                                                                          'Matemáticas '
                                                                                                                          'Aplicadas '
                                                                                                                          'y '
                                                                                                                          'en '
                                                                                                                          'Sistemas-UNAM, '
                                                                                                                          'Mexico',
                                                                                                                          ''),
 'Instituto de Matemáticas, UNAM, Mexico | IRIF-Université Paris Cité': ('Instituto de Matemáticas, UNAM, Mexico', ''),
 'Irif and Instituto de Matemàticas, Unam, Mexico City, Mexico': ('Instituto de Matemàticas, Unam, Mexico City, Mexico',
                                                                  ''),
 'Laboratorio Nacional de Análisis y Síntesis Ecológica, Escuela Nacional de Estudios Superiores, UNAM Unidad Morelia, 58190 Morelia, Michoacán, Mexico | Centro de Investigaciones en Geografía Ambiental (CIGA), UNAM Unidad Morelia, 58190 Morelia, Michoacán, Mexico': ('Laboratorio '
                                                                                                                                                                                                                                                                            'Nacional '
                                                                                                                                                                                                                                                                            'de '
                                                                                                                                                                                                                                                                            'Análisis '
                                                                                                                                                                                                                                                                            'y '
                                                                                                                                                                                                                                                                            'Síntesis '
                                                                                                                                                                                                                                                                            'Ecológica, '
                                                                                                                                                                                                                                                                            'Escuela '
                                                                                                                                                                                                                                                                            'Nacional '
                                                                                                                                                                                                                                                                            'de '
                                                                                                                                                                                                                                                                            'Estudios '
                                                                                                                                                                                                                                                                            'Superiores, '
                                                                                                                                                                                                                                                                            'UNAM '
                                                                                                                                                                                                                                                                            'Unidad '
                                                                                                                                                                                                                                                                            'Morelia, '
                                                                                                                                                                                                                                                                            '58190 '
                                                                                                                                                                                                                                                                            'Morelia, '
                                                                                                                                                                                                                                                                            'Michoacán, '
                                                                                                                                                                                                                                                                            'Mexico',
                                                                                                                                                                                                                                                                            'Centro '
                                                                                                                                                                                                                                                                            'de '
                                                                                                                                                                                                                                                                            'Investigaciones '
                                                                                                                                                                                                                                                                            'en '
                                                                                                                                                                                                                                                                            'Geografía '
                                                                                                                                                                                                                                                                            'Ambiental '
                                                                                                                                                                                                                                                                            '(CIGA), '
                                                                                                                                                                                                                                                                            'UNAM '
                                                                                                                                                                                                                                                                            'Unidad '
                                                                                                                                                                                                                                                                            'Morelia, '
                                                                                                                                                                                                                                                                            '58190 '
                                                                                                                                                                                                                                                                            'Morelia, '
                                                                                                                                                                                                                                                                            'Michoacán, '
                                                                                                                                                                                                                                                                            'Mexico'),
 'Laboratorio Nacional de Análisis y Síntesis Ecológica, Escuela Nacional de Estudios Superiores, UNAM Unidad Morelia, 58190 Morelia, Michoacán, Mexico | Environmental Geography Group, Institute for Environmental Studies (IVM), Vrije Universiteit Amsterdam, 1081 HV Amsterdam, The Netherlands': ('Laboratorio '
                                                                                                                                                                                                                                                                                                        'Nacional '
                                                                                                                                                                                                                                                                                                        'de '
                                                                                                                                                                                                                                                                                                        'Análisis '
                                                                                                                                                                                                                                                                                                        'y '
                                                                                                                                                                                                                                                                                                        'Síntesis '
                                                                                                                                                                                                                                                                                                        'Ecológica, '
                                                                                                                                                                                                                                                                                                        'Escuela '
                                                                                                                                                                                                                                                                                                        'Nacional '
                                                                                                                                                                                                                                                                                                        'de '
                                                                                                                                                                                                                                                                                                        'Estudios '
                                                                                                                                                                                                                                                                                                        'Superiores, '
                                                                                                                                                                                                                                                                                                        'UNAM '
                                                                                                                                                                                                                                                                                                        'Unidad '
                                                                                                                                                                                                                                                                                                        'Morelia, '
                                                                                                                                                                                                                                                                                                        '58190 '
                                                                                                                                                                                                                                                                                                        'Morelia, '
                                                                                                                                                                                                                                                                                                        'Michoacán, '
                                                                                                                                                                                                                                                                                                        'Mexico',
                                                                                                                                                                                                                                                                                                        ''),
 'Laboratório Nacional de Computação Científica - LNCC, Avenida Getúlio Vargas, Petrópolis, 25651075, Rio de Janeiro, Brazil, Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas, Universidad Nacional Autónoma de México, Unidad Académica del Estado de Yucatán, Carretera Sierra Papacal, Mérida 97302, Yucatán, México': ('Instituto '
                                                                                                                                                                                                                                                                                                                                               'de '
                                                                                                                                                                                                                                                                                                                                               'Investigaciones '
                                                                                                                                                                                                                                                                                                                                               'en '
                                                                                                                                                                                                                                                                                                                                               'Matemáticas '
                                                                                                                                                                                                                                                                                                                                               'Aplicadas '
                                                                                                                                                                                                                                                                                                                                               'y '
                                                                                                                                                                                                                                                                                                                                               'en '
                                                                                                                                                                                                                                                                                                                                               'Sistemas, '
                                                                                                                                                                                                                                                                                                                                               'Universidad '
                                                                                                                                                                                                                                                                                                                                               'Nacional '
                                                                                                                                                                                                                                                                                                                                               'Autónoma '
                                                                                                                                                                                                                                                                                                                                               'de '
                                                                                                                                                                                                                                                                                                                                               'México, '
                                                                                                                                                                                                                                                                                                                                               'Unidad '
                                                                                                                                                                                                                                                                                                                                               'Académica '
                                                                                                                                                                                                                                                                                                                                               'del '
                                                                                                                                                                                                                                                                                                                                               'Estado '
                                                                                                                                                                                                                                                                                                                                               'de '
                                                                                                                                                                                                                                                                                                                                               'Yucatán, '
                                                                                                                                                                                                                                                                                                                                               'Carretera '
                                                                                                                                                                                                                                                                                                                                               'Sierra '
                                                                                                                                                                                                                                                                                                                                               'Papacal, '
                                                                                                                                                                                                                                                                                                                                               'Mérida '
                                                                                                                                                                                                                                                                                                                                               '97302, '
                                                                                                                                                                                                                                                                                                                                               'Yucatán, '
                                                                                                                                                                                                                                                                                                                                               'México',
                                                                                                                                                                                                                                                                                                                                               ''),
 'Laboratório Nacional de Computação Científica - LNCC, Avenida Getúlio Vargas, Petrópolis, Rio de Janeiro, 25651075, Brazil, Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas, Universidad Nacional Autónoma de México, Unidad Académica del Estado de Yucatán, Carretera Sierra Papacal, Mérida, Yucatán, 97302, Mexico': ('Instituto '
                                                                                                                                                                                                                                                                                                                                                'de '
                                                                                                                                                                                                                                                                                                                                                'Investigaciones '
                                                                                                                                                                                                                                                                                                                                                'en '
                                                                                                                                                                                                                                                                                                                                                'Matemáticas '
                                                                                                                                                                                                                                                                                                                                                'Aplicadas '
                                                                                                                                                                                                                                                                                                                                                'y '
                                                                                                                                                                                                                                                                                                                                                'en '
                                                                                                                                                                                                                                                                                                                                                'Sistemas, '
                                                                                                                                                                                                                                                                                                                                                'Universidad '
                                                                                                                                                                                                                                                                                                                                                'Nacional '
                                                                                                                                                                                                                                                                                                                                                'Autónoma '
                                                                                                                                                                                                                                                                                                                                                'de '
                                                                                                                                                                                                                                                                                                                                                'México, '
                                                                                                                                                                                                                                                                                                                                                'Unidad '
                                                                                                                                                                                                                                                                                                                                                'Académica '
                                                                                                                                                                                                                                                                                                                                                'del '
                                                                                                                                                                                                                                                                                                                                                'Estado '
                                                                                                                                                                                                                                                                                                                                                'de '
                                                                                                                                                                                                                                                                                                                                                'Yucatán, '
                                                                                                                                                                                                                                                                                                                                                'Carretera '
                                                                                                                                                                                                                                                                                                                                                'Sierra '
                                                                                                                                                                                                                                                                                                                                                'Papacal, '
                                                                                                                                                                                                                                                                                                                                                'Mérida, '
                                                                                                                                                                                                                                                                                                                                                'Yucatán, '
                                                                                                                                                                                                                                                                                                                                                '97302, '
                                                                                                                                                                                                                                                                                                                                                'Mexico',
                                                                                                                                                                                                                                                                                                                                                ''),
 'Mathematical Robotics Science Division, Sirius University of Science and Technology, 1 Olympic Av., Sirius, 354340, Russian Federation, Facultad de Ingenieria, Universidad Nacional Autónoma de México (UNAM), Coyoacan, Ciudad Universitaria, Mexico City, 04510, Mexico': ('Facultad '
                                                                                                                                                                                                                                                                                'de '
                                                                                                                                                                                                                                                                                'Ingenieria, '
                                                                                                                                                                                                                                                                                'Universidad '
                                                                                                                                                                                                                                                                                'Nacional '
                                                                                                                                                                                                                                                                                'Autónoma '
                                                                                                                                                                                                                                                                                'de '
                                                                                                                                                                                                                                                                                'México '
                                                                                                                                                                                                                                                                                '(UNAM), '
                                                                                                                                                                                                                                                                                'Coyoacan, '
                                                                                                                                                                                                                                                                                'Ciudad '
                                                                                                                                                                                                                                                                                'Universitaria, '
                                                                                                                                                                                                                                                                                'Mexico '
                                                                                                                                                                                                                                                                                'City, '
                                                                                                                                                                                                                                                                                '04510, '
                                                                                                                                                                                                                                                                                'Mexico',
                                                                                                                                                                                                                                                                                ''),
 'Morelia Institute of Technology, Mexico | Instituto de Investigaciones en Ecosistemas y Sustentabilidad, Mexico': ('Instituto '
                                                                                                                     'de '
                                                                                                                     'Investigaciones '
                                                                                                                     'en '
                                                                                                                     'Ecosistemas '
                                                                                                                     'y '
                                                                                                                     'Sustentabilidad, '
                                                                                                                     'Mexico',
                                                                                                                     ''),
 'National Laboratory on Health, Molecular Diagnostics and Environmental Effects on Chronic-Degenerative Diseases, Faculty of Higher Studies Iztacala, UNAM, Avenida de los Barrios #1, Los Reyes Iztacala, Mexico State, Tlanepantla, 54090, Mexico, Biomedicine Unit, Faculty of Higher Studies Iztacala, UNAM, Avenida de los Barrios #1, Los Reyes Iztacala, Mexico Sstate, Tlalnepantla, 54090, Mexico': ('National '
                                                                                                                                                                                                                                                                                                                                                                                                               'Laboratory '
                                                                                                                                                                                                                                                                                                                                                                                                               'on '
                                                                                                                                                                                                                                                                                                                                                                                                               'Health, '
                                                                                                                                                                                                                                                                                                                                                                                                               'Molecular '
                                                                                                                                                                                                                                                                                                                                                                                                               'Diagnostics '
                                                                                                                                                                                                                                                                                                                                                                                                               'and '
                                                                                                                                                                                                                                                                                                                                                                                                               'Environmental '
                                                                                                                                                                                                                                                                                                                                                                                                               'Effects '
                                                                                                                                                                                                                                                                                                                                                                                                               'on '
                                                                                                                                                                                                                                                                                                                                                                                                               'Chronic-Degenerative '
                                                                                                                                                                                                                                                                                                                                                                                                               'Diseases, '
                                                                                                                                                                                                                                                                                                                                                                                                               'Faculty '
                                                                                                                                                                                                                                                                                                                                                                                                               'of '
                                                                                                                                                                                                                                                                                                                                                                                                               'Higher '
                                                                                                                                                                                                                                                                                                                                                                                                               'Studies '
                                                                                                                                                                                                                                                                                                                                                                                                               'Iztacala, '
                                                                                                                                                                                                                                                                                                                                                                                                               'UNAM, '
                                                                                                                                                                                                                                                                                                                                                                                                               'Avenida '
                                                                                                                                                                                                                                                                                                                                                                                                               'de '
                                                                                                                                                                                                                                                                                                                                                                                                               'los '
                                                                                                                                                                                                                                                                                                                                                                                                               'Barrios '
                                                                                                                                                                                                                                                                                                                                                                                                               '#1, '
                                                                                                                                                                                                                                                                                                                                                                                                               'Los '
                                                                                                                                                                                                                                                                                                                                                                                                               'Reyes '
                                                                                                                                                                                                                                                                                                                                                                                                               'Iztacala, '
                                                                                                                                                                                                                                                                                                                                                                                                               'Mexico '
                                                                                                                                                                                                                                                                                                                                                                                                               'State, '
                                                                                                                                                                                                                                                                                                                                                                                                               'Tlanepantla, '
                                                                                                                                                                                                                                                                                                                                                                                                               '54090, '
                                                                                                                                                                                                                                                                                                                                                                                                               'Mexico',
                                                                                                                                                                                                                                                                                                                                                                                                               'Biomedicine '
                                                                                                                                                                                                                                                                                                                                                                                                               'Unit, '
                                                                                                                                                                                                                                                                                                                                                                                                               'Faculty '
                                                                                                                                                                                                                                                                                                                                                                                                               'of '
                                                                                                                                                                                                                                                                                                                                                                                                               'Higher '
                                                                                                                                                                                                                                                                                                                                                                                                               'Studies '
                                                                                                                                                                                                                                                                                                                                                                                                               'Iztacala, '
                                                                                                                                                                                                                                                                                                                                                                                                               'UNAM, '
                                                                                                                                                                                                                                                                                                                                                                                                               'Avenida '
                                                                                                                                                                                                                                                                                                                                                                                                               'de '
                                                                                                                                                                                                                                                                                                                                                                                                               'los '
                                                                                                                                                                                                                                                                                                                                                                                                               'Barrios '
                                                                                                                                                                                                                                                                                                                                                                                                               '#1, '
                                                                                                                                                                                                                                                                                                                                                                                                               'Los '
                                                                                                                                                                                                                                                                                                                                                                                                               'Reyes '
                                                                                                                                                                                                                                                                                                                                                                                                               'Iztacala, '
                                                                                                                                                                                                                                                                                                                                                                                                               'Mexico '
                                                                                                                                                                                                                                                                                                                                                                                                               'Sstate, '
                                                                                                                                                                                                                                                                                                                                                                                                               'Tlalnepantla, '
                                                                                                                                                                                                                                                                                                                                                                                                               '54090, '
                                                                                                                                                                                                                                                                                                                                                                                                               'Mexico'),
 'Northeastern Univ, Boston, MA 02115 USA. | Univ Natl Autonoma Mexico UNAM, Mexico City, DF, Mexico.': ('Univ Natl '
                                                                                                         'Autonoma '
                                                                                                         'Mexico UNAM, '
                                                                                                         'Mexico City, '
                                                                                                         'DF, Mexico.',
                                                                                                         ''),
 'Northeastern University, Boston, MA, United States, Universidad Nacional Autonoma de Mexico, Mexico City, Mexico': ('Universidad '
                                                                                                                      'Nacional '
                                                                                                                      'Autonoma '
                                                                                                                      'de '
                                                                                                                      'Mexico, '
                                                                                                                      'Mexico '
                                                                                                                      'City, '
                                                                                                                      'Mexico',
                                                                                                                      ''),
 'Northeastern University, Boston, Massachusetts, USA | Universidad Nacional Autonoma de Mexico, Mexico City, Mexico': ('Universidad '
                                                                                                                        'Nacional '
                                                                                                                        'Autonoma '
                                                                                                                        'de '
                                                                                                                        'Mexico, '
                                                                                                                        'Mexico '
                                                                                                                        'City, '
                                                                                                                        'Mexico',
                                                                                                                        ''),
 'Northeastern University, Boston; MA, United States | Universidad Nacional Autonoma de Mexico, Mexico City, Mexico': ('Universidad '
                                                                                                                       'Nacional '
                                                                                                                       'Autonoma '
                                                                                                                       'de '
                                                                                                                       'Mexico, '
                                                                                                                       'Mexico '
                                                                                                                       'City, '
                                                                                                                       'Mexico',
                                                                                                                       ''),
 'Northeastern University, USA | Universidad Nacional Autonoma de Mexico (UNAM), Mexico': ('Universidad Nacional '
                                                                                           'Autonoma de Mexico (UNAM), '
                                                                                           'Mexico',
                                                                                           ''),
 'Northeastern University, USA | Universidad Nacional Autónoma de México (UNAM), Mexico': ('Universidad Nacional '
                                                                                           'Autónoma de México (UNAM), '
                                                                                           'Mexico',
                                                                                           ''),
 'Northeastern University, United States, Universidad Nacional Autonoma de Mexico (UNAM), United States': ('Universidad '
                                                                                                           'Nacional '
                                                                                                           'Autonoma '
                                                                                                           'de Mexico '
                                                                                                           '(UNAM), '
                                                                                                           'United '
                                                                                                           'States',
                                                                                                           ''),
 'Posgrado en Ciencias de la Tierra, Universidad Nacional Autónoma de México, C.P. 04510, Ciudad de México, México | Geociencias Aplicadas, Instituto Potosino de Investigación Científica y Tecnológica, C.P. 78216, San Luis Potosí, SLP, México': ('Posgrado '
                                                                                                                                                                                                                                                      'en '
                                                                                                                                                                                                                                                      'Ciencias '
                                                                                                                                                                                                                                                      'de '
                                                                                                                                                                                                                                                      'la '
                                                                                                                                                                                                                                                      'Tierra, '
                                                                                                                                                                                                                                                      'Universidad '
                                                                                                                                                                                                                                                      'Nacional '
                                                                                                                                                                                                                                                      'Autónoma '
                                                                                                                                                                                                                                                      'de '
                                                                                                                                                                                                                                                      'México, '
                                                                                                                                                                                                                                                      'C.P. '
                                                                                                                                                                                                                                                      '04510, '
                                                                                                                                                                                                                                                      'Ciudad '
                                                                                                                                                                                                                                                      'de '
                                                                                                                                                                                                                                                      'México, '
                                                                                                                                                                                                                                                      'México',
                                                                                                                                                                                                                                                      ''),
 'Posgrado en Ciencias de la Tierra, Universidad Nacional Autónoma de México, Ciudad de México, C.P. 04510, Mexico, Geociencias Aplicadas, Instituto Potosino de Investigación Científica y Tecnológica, SLP, San Luis Potosí, C.P. 78216, Mexico': ('Posgrado '
                                                                                                                                                                                                                                                     'en '
                                                                                                                                                                                                                                                     'Ciencias '
                                                                                                                                                                                                                                                     'de '
                                                                                                                                                                                                                                                     'la '
                                                                                                                                                                                                                                                     'Tierra, '
                                                                                                                                                                                                                                                     'Universidad '
                                                                                                                                                                                                                                                     'Nacional '
                                                                                                                                                                                                                                                     'Autónoma '
                                                                                                                                                                                                                                                     'de '
                                                                                                                                                                                                                                                     'México, '
                                                                                                                                                                                                                                                     'Ciudad '
                                                                                                                                                                                                                                                     'de '
                                                                                                                                                                                                                                                     'México, '
                                                                                                                                                                                                                                                     'C.P. '
                                                                                                                                                                                                                                                     '04510, '
                                                                                                                                                                                                                                                     'Mexico',
                                                                                                                                                                                                                                                     ''),
 'Red de Investigación OAC Optimización, Automatización y Control, Queretaro, El Marqués, 76240, Mexico, Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas, Unidad Académica del Estado de Yucatán, Universidad Nacional Autónoma de México, Mérida, 97357, Mexico': ('Instituto '
                                                                                                                                                                                                                                                                                        'de '
                                                                                                                                                                                                                                                                                        'Investigaciones '
                                                                                                                                                                                                                                                                                        'en '
                                                                                                                                                                                                                                                                                        'Matemáticas '
                                                                                                                                                                                                                                                                                        'Aplicadas '
                                                                                                                                                                                                                                                                                        'y '
                                                                                                                                                                                                                                                                                        'en '
                                                                                                                                                                                                                                                                                        'Sistemas, '
                                                                                                                                                                                                                                                                                        'Unidad '
                                                                                                                                                                                                                                                                                        'Académica '
                                                                                                                                                                                                                                                                                        'del '
                                                                                                                                                                                                                                                                                        'Estado '
                                                                                                                                                                                                                                                                                        'de '
                                                                                                                                                                                                                                                                                        'Yucatán, '
                                                                                                                                                                                                                                                                                        'Universidad '
                                                                                                                                                                                                                                                                                        'Nacional '
                                                                                                                                                                                                                                                                                        'Autónoma '
                                                                                                                                                                                                                                                                                        'de '
                                                                                                                                                                                                                                                                                        'México, '
                                                                                                                                                                                                                                                                                        'Mérida, '
                                                                                                                                                                                                                                                                                        '97357, '
                                                                                                                                                                                                                                                                                        'Mexico',
                                                                                                                                                                                                                                                                                        ''),
 'SECIHTI - Instituto de Ciencias Aplicadas y Tecnología, Universidad Nacional Autónoma de México, Ciudad de México, 04510, Mexico, SECIHTI - CICESE, Alianza Centro #540, PIIT Apodaca, Nuevo León, Monterrey, 66629, Mexico': ('SECIHTI '
                                                                                                                                                                                                                                 '- '
                                                                                                                                                                                                                                 'Instituto '
                                                                                                                                                                                                                                 'de '
                                                                                                                                                                                                                                 'Ciencias '
                                                                                                                                                                                                                                 'Aplicadas '
                                                                                                                                                                                                                                 'y '
                                                                                                                                                                                                                                 'Tecnología, '
                                                                                                                                                                                                                                 'Universidad '
                                                                                                                                                                                                                                 'Nacional '
                                                                                                                                                                                                                                 'Autónoma '
                                                                                                                                                                                                                                 'de '
                                                                                                                                                                                                                                 'México, '
                                                                                                                                                                                                                                 'Ciudad '
                                                                                                                                                                                                                                 'de '
                                                                                                                                                                                                                                 'México, '
                                                                                                                                                                                                                                 '04510, '
                                                                                                                                                                                                                                 'Mexico',
                                                                                                                                                                                                                                 ''),
 'School of Mathematics, University of the Witwatersrand, Private Bag X3, Wits 2050, Johannesburg, South Africa, Centro de Ciencias Matemáticas, UNAM, Morelia, Mexico': ('Centro '
                                                                                                                                                                          'de '
                                                                                                                                                                          'Ciencias '
                                                                                                                                                                          'Matemáticas, '
                                                                                                                                                                          'UNAM, '
                                                                                                                                                                          'Morelia, '
                                                                                                                                                                          'Mexico',
                                                                                                                                                                          ''),
 'Sirius University of Science and Technology, Mathematical Robotics Science Division, 1 Olympic Av., Russia | Universidad Nacional Autónoma de México, Mexico': ('Universidad '
                                                                                                                                                                    'Nacional '
                                                                                                                                                                    'Autónoma '
                                                                                                                                                                    'de '
                                                                                                                                                                    'México, '
                                                                                                                                                                    'Mexico',
                                                                                                                                                                    ''),
 'State University of New York at Binghamton, School of Systems Science and Industrial Engineering, Universidad Nacional Autónoma de México, Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas, Centro de Ciencias de la Complejidad.': ('Universidad '
                                                                                                                                                                                                                                                           'Nacional '
                                                                                                                                                                                                                                                           'Autónoma '
                                                                                                                                                                                                                                                           'de '
                                                                                                                                                                                                                                                           'México, '
                                                                                                                                                                                                                                                           'Instituto '
                                                                                                                                                                                                                                                           'de '
                                                                                                                                                                                                                                                           'Investigaciones '
                                                                                                                                                                                                                                                           'en '
                                                                                                                                                                                                                                                           'Matemáticas '
                                                                                                                                                                                                                                                           'Aplicadas '
                                                                                                                                                                                                                                                           'y '
                                                                                                                                                                                                                                                           'en '
                                                                                                                                                                                                                                                           'Sistemas, '
                                                                                                                                                                                                                                                           'Centro '
                                                                                                                                                                                                                                                           'de '
                                                                                                                                                                                                                                                           'Ciencias '
                                                                                                                                                                                                                                                           'de '
                                                                                                                                                                                                                                                           'la '
                                                                                                                                                                                                                                                           'Complejidad.',
                                                                                                                                                                                                                                                           ''),
 'Universidad Nacional Autonoma de Mexico, Instituto de Neurobiologia, Mexico | University of Groningen, Department of Biomedical Sciences of Cells and Systems, Netherlands': ('Universidad '
                                                                                                                                                                                'Nacional '
                                                                                                                                                                                'Autonoma '
                                                                                                                                                                                'de '
                                                                                                                                                                                'Mexico, '
                                                                                                                                                                                'Instituto '
                                                                                                                                                                                'de '
                                                                                                                                                                                'Neurobiologia, '
                                                                                                                                                                                'Mexico',
                                                                                                                                                                                ''),
 "Universidad Nacional Autonoma de Mexico, Physica Institute, Mexico | National Institute of Phsychiatry 'Ramón de la Fuente Muñiz', Mexico": ('Universidad '
                                                                                                                                                 'Nacional '
                                                                                                                                                 'Autonoma '
                                                                                                                                                 'de '
                                                                                                                                                 'Mexico, '
                                                                                                                                                 'Physica '
                                                                                                                                                 'Institute, '
                                                                                                                                                 'Mexico',
                                                                                                                                                 ''),
 'Universidad Nacional Autónoma de México, Mexico City, Mexico | Departamento de Procesamiento de Señales, Facultad de Ingeniería, Universidad Nacional Autónoma de México, Mexico City, Mexico': ('Universidad '
                                                                                                                                                                                                   'Nacional '
                                                                                                                                                                                                   'Autónoma '
                                                                                                                                                                                                   'de '
                                                                                                                                                                                                   'México, '
                                                                                                                                                                                                   'Mexico '
                                                                                                                                                                                                   'City, '
                                                                                                                                                                                                   'Mexico',
                                                                                                                                                                                                   'Departamento '
                                                                                                                                                                                                   'de '
                                                                                                                                                                                                   'Procesamiento '
                                                                                                                                                                                                   'de '
                                                                                                                                                                                                   'Señales, '
                                                                                                                                                                                                   'Facultad '
                                                                                                                                                                                                   'de '
                                                                                                                                                                                                   'Ingeniería, '
                                                                                                                                                                                                   'Universidad '
                                                                                                                                                                                                   'Nacional '
                                                                                                                                                                                                   'Autónoma '
                                                                                                                                                                                                   'de '
                                                                                                                                                                                                   'México, '
                                                                                                                                                                                                   'Mexico '
                                                                                                                                                                                                   'City, '
                                                                                                                                                                                                   'Mexico'),
 'Universidad Nacional Autónoma de México, Mexico City, Mexico, Departamento de Procesamiento de Señales, Facultad de Ingeniería, Universidad Nacional Autónoma de México, Mexico City, Mexico': ('Universidad '
                                                                                                                                                                                                  'Nacional '
                                                                                                                                                                                                  'Autónoma '
                                                                                                                                                                                                  'de '
                                                                                                                                                                                                  'México, '
                                                                                                                                                                                                  'Mexico '
                                                                                                                                                                                                  'City, '
                                                                                                                                                                                                  'Mexico',
                                                                                                                                                                                                  'Departamento '
                                                                                                                                                                                                  'de '
                                                                                                                                                                                                  'Procesamiento '
                                                                                                                                                                                                  'de '
                                                                                                                                                                                                  'Señales, '
                                                                                                                                                                                                  'Facultad '
                                                                                                                                                                                                  'de '
                                                                                                                                                                                                  'Ingeniería, '
                                                                                                                                                                                                  'Universidad '
                                                                                                                                                                                                  'Nacional '
                                                                                                                                                                                                  'Autónoma '
                                                                                                                                                                                                  'de '
                                                                                                                                                                                                  'México, '
                                                                                                                                                                                                  'Mexico '
                                                                                                                                                                                                  'City, '
                                                                                                                                                                                                  'Mexico'),
 'Universidad Nacional Autónoma de México, Mexico | Instituto de Ingeniería, Mexico': ('Universidad Nacional Autónoma '
                                                                                       'de México, Mexico',
                                                                                       'Instituto de Ingeniería, '
                                                                                       'Mexico'),
 'Universidad Nacional Autónoma de México, Mexico | Instituto de Investigaciones en Matemáticas Aplicadas y en Sistemas, Mexico': ('Universidad '
                                                                                                                                   'Nacional '
                                                                                                                                   'Autónoma '
                                                                                                                                   'de '
                                                                                                                                   'México, '
                                                                                                                                   'Mexico',
                                                                                                                                   'Instituto '
                                                                                                                                   'de '
                                                                                                                                   'Investigaciones '
                                                                                                                                   'en '
                                                                                                                                   'Matemáticas '
                                                                                                                                   'Aplicadas '
                                                                                                                                   'y '
                                                                                                                                   'en '
                                                                                                                                   'Sistemas, '
                                                                                                                                   'Mexico'),
 'Universidad Nacional Autónoma de México, Mexico | Posgrado en Ciencia e Ingeniería de la Computación, Mexico': ('Universidad '
                                                                                                                  'Nacional '
                                                                                                                  'Autónoma '
                                                                                                                  'de '
                                                                                                                  'México, '
                                                                                                                  'Mexico',
                                                                                                                  'Posgrado '
                                                                                                                  'en '
                                                                                                                  'Ciencia '
                                                                                                                  'e '
                                                                                                                  'Ingeniería '
                                                                                                                  'de '
                                                                                                                  'la '
                                                                                                                  'Computación, '
                                                                                                                  'Mexico'),
 'Universidad Nacional Autónoma de México, Universitat de Barcelona, Spain': ('Universidad Nacional Autónoma de México',
                                                                              '')}

print("Afiliaciones con ajuste manual exacto:", len(ajustes_afiliaciones))


Afiliaciones con ajuste manual exacto: 76


## 4. Aplicar la clasificación

In [ ]:
estado_por_afiliacion = dict(
    zip(
        clasificacion["Afiliacion_original"],
        clasificacion["Estado_UNAM"]
    )
)


# Copia de trabajo.
df_trabajo = df.copy()

df_trabajo["Estado_temporal"] = (
    df_trabajo["Afiliacion1"]
    .map(estado_por_afiliacion)
)


if df_trabajo["Estado_temporal"].isna().any():
    raise ValueError(
        "Existen filas sin clasificación."
    )


conteos = df_trabajo[
    "Estado_temporal"
].value_counts()


print("UNAM:", conteos.get("UNAM", 0))
print("EXTERNO:", conteos.get("EXTERNO", 0))
print("REVISAR:", conteos.get("REVISAR", 0))
print("TOTAL:", len(df_trabajo))


UNAM: 4152
EXTERNO: 5363
REVISAR: 0
TOTAL: 9515


## 5. Crear la salida con autores UNAM

Solo se modifican `Afiliacion1` y `Afiliacion2` cuando el caso requiere separar o retirar una afiliación externa.

In [ ]:
autores_unam = (
    df_trabajo[
        df_trabajo["Estado_temporal"] == "UNAM"
    ][columnas_canonicas]
    .copy()
    .reset_index(drop=True)
)


for i in autores_unam.index:

    afiliacion_original = (
        autores_unam.at[i, "Afiliacion1"]
    )

    if afiliacion_original in ajustes_afiliaciones:

        afiliacion_1, afiliacion_2 = (
            ajustes_afiliaciones[
                afiliacion_original
            ]
        )

        autores_unam.at[
            i, "Afiliacion1"
        ] = afiliacion_1

        autores_unam.at[
            i, "Afiliacion2"
        ] = afiliacion_2

    else:
        # Si no requiere ajuste,
        # se conserva Afiliacion1 original
        # y Afiliacion2 permanece vacío.
        autores_unam.at[
            i, "Afiliacion2"
        ] = ""


revision = (
    df_trabajo[
        df_trabajo["Estado_temporal"] == "REVISAR"
    ][columnas_canonicas]
    .copy()
    .reset_index(drop=True)
)


print("Relaciones UNAM:", len(autores_unam))
print("Relaciones pendientes de revisión:", len(revision))


Relaciones UNAM: 4152
Relaciones pendientes de revisión: 0


## 6. Guardar resultados

In [ ]:
autores_unam.to_csv(
    archivo_unam,
    index=False,
    encoding="utf-8-sig"
)


if len(revision) > 0:

    revision.to_csv(
        archivo_revision,
        index=False,
        encoding="utf-8-sig"
    )

else:

    # Evitar que quede un archivo viejo
    # de una ejecución anterior.
    if os.path.exists(archivo_revision):
        os.remove(archivo_revision)


print("Guardado:", archivo_unam)

if len(revision) > 0:
    print("Guardado:", archivo_revision)
else:
    print("No fue necesario crear afiliaciones_revision.csv")


Guardado: ../04_Limpieza/01_Internos_unam/autores_unam_identificados.csv
No fue necesario crear afiliaciones_revision.csv


## 7. Validaciones finales

In [ ]:
hash_original_despues = sha256(archivo)


# --------------------------------------------------
# 1. Archivo original intacto
# --------------------------------------------------

assert (
    hash_original_antes
    == hash_original_despues
)


# --------------------------------------------------
# 2. Las salidas conservan 16 columnas
# --------------------------------------------------

assert (
    list(autores_unam.columns)
    == columnas_canonicas
)

assert (
    list(revision.columns)
    == columnas_canonicas
)


# --------------------------------------------------
# 3. Todas las relaciones fueron clasificadas
# --------------------------------------------------

assert (
    conteos.get("UNAM", 0)
    + conteos.get("EXTERNO", 0)
    + conteos.get("REVISAR", 0)
    == len(df)
)


# --------------------------------------------------
# 4. La salida tiene únicamente relaciones UNAM
# --------------------------------------------------

assert (
    len(autores_unam)
    == conteos.get("UNAM", 0)
)


# --------------------------------------------------
# 5. Revisar contiene solamente REVISAR
# --------------------------------------------------

assert (
    len(revision)
    == conteos.get("REVISAR", 0)
)


# --------------------------------------------------
# 6. No modificar campos bibliográficos
# --------------------------------------------------

campos_intocables = [
    "Base_origen",
    "Fuente_origen",
    "indice",
    "Titulo",
    "Año",
    "Autor_norm",
    "ISBN",
    "ISSN",
    "Doi",
    "URL",
    "Area",
    "SubArea",
    "Keywords",
    "Abstract",
]


originales_unam = (
    df_trabajo[
        df_trabajo["Estado_temporal"] == "UNAM"
    ][columnas_canonicas]
    .reset_index(drop=True)
)


for columna in campos_intocables:

    if not originales_unam[
        columna
    ].equals(
        autores_unam[columna]
    ):
        raise ValueError(
            "Se modificó el campo: "
            + columna
        )


# --------------------------------------------------
# 7. No debe quedar " | " dentro de una celda
# --------------------------------------------------

assert not autores_unam[
    "Afiliacion1"
].str.contains(
    r" \| ",
    regex=True
).any()

assert not autores_unam[
    "Afiliacion2"
].str.contains(
    r" \| ",
    regex=True
).any()


# --------------------------------------------------
# 8. No crear filas nuevas
# --------------------------------------------------

columnas_clave = campos_intocables


claves_originales = Counter(
    map(
        tuple,
        originales_unam[
            columnas_clave
        ].to_numpy()
    )
)


claves_salida = Counter(
    map(
        tuple,
        autores_unam[
            columnas_clave
        ].to_numpy()
    )
)


assert (
    claves_originales
    == claves_salida
)


print("Todas las validaciones fueron correctas.")


Todas las validaciones fueron correctas.


## 8. Resumen final

In [ ]:
conteo_unicas = (
    clasificacion[
        "Estado_UNAM"
    ]
    .value_counts()
)


print("Total de relaciones:", len(df))
print()

print(
    "UNAM:",
    conteos.get("UNAM", 0),
    "relaciones"
)

print(
    "EXTERNO:",
    conteos.get("EXTERNO", 0),
    "relaciones"
)

print(
    "REVISAR:",
    conteos.get("REVISAR", 0),
    "relaciones"
)

print()

print(
    "Afiliaciones únicas:",
    len(clasificacion)
)

print(
    "Afiliaciones únicas UNAM:",
    conteo_unicas.get("UNAM", 0)
)

print(
    "Afiliaciones únicas EXTERNO:",
    conteo_unicas.get("EXTERNO", 0)
)

print(
    "Afiliaciones únicas REVISAR:",
    conteo_unicas.get("REVISAR", 0)
)

print()

print(
    "Archivo original intacto:",
    hash_original_antes
    == hash_original_despues
)


if len(revision) > 0:
    display(revision)
else:
    print("No existen casos pendientes de revisión.")


Total de relaciones: 9515

UNAM: 4152 relaciones
EXTERNO: 5363 relaciones
REVISAR: 0 relaciones

Afiliaciones únicas: 1604
Afiliaciones únicas UNAM: 695
Afiliaciones únicas EXTERNO: 909
Afiliaciones únicas REVISAR: 0

Archivo original intacto: True
No existen casos pendientes de revisión.
